In [1]:
from dotenv import load_dotenv
from pathlib import Path
load_dotenv()

True

In [2]:
from src.ASRData import ASRData
from src.metrics import calculate_csr_metrics
%load_ext autoreload
%autoreload 2

Test API

In [4]:
from openai import OpenAI
client = OpenAI()

In [3]:
response = client.responses.create(
    model="gpt-5-nano",
    input="Write a short description of speach to text models."
)

print(response.output_text)

Speech-to-text models convert spoken language into written text. They process audio, extract features, and decode content using acoustic and language models—often end-to-end neural networks like Transformers. They enable transcription, captions, and voice assistants, but face challenges such as noise, accents, and real-time latency.


Load audio file and transcribe with different models.
Correct test transcription:
- Hello I'm Jane!
- Cześć Jane, mam na imię Michał!

In [6]:
audio_file = open("../data/example_1/audio.mp3", "rb")

Model: Whisper-1

In [5]:
transcription = client.audio.transcriptions.create(
    file=audio_file,
    model="whisper-1",
    response_format="text"
)

In [6]:
print(transcription)

Cześć! Nazywam się Jane.



Model: gpt-4o-transcribe

In [7]:
transcription_2 = client.audio.transcriptions.create(
    file=audio_file,
    model="gpt-4o-transcribe",
    response_format="text"
)

In [8]:
print(transcription_2)

Cześć Jane, mam na imię Michał.



Model: gpt-4o-transcribe-diarize

In [9]:
transcription_3 = client.audio.transcriptions.create(
    file=audio_file,
    model="gpt-4o-transcribe-diarize",
    response_format="text"
)

In [10]:
print(transcription_3)

Hello, I'm Jane. Cześć Jane, mam na imię Michał.



Diarized json output

In [7]:
transcription_4 = client.audio.transcriptions.create(
    file=audio_file,
    model="gpt-4o-transcribe-diarize",
    response_format="diarized_json"
)

In [16]:
transcription_4.segments

[TranscriptionDiarizedSegment(id='seg_0', end=0.30000000000000004, speaker='A', start=0.0, text=' Hello,', type='transcript.text.segment'),
 TranscriptionDiarizedSegment(id='seg_1', end=1.05, speaker='A', start=0.55, text=" I'm Jane.", type='transcript.text.segment'),
 TranscriptionDiarizedSegment(id='seg_2', end=3.3499999999999996, speaker='B', start=1.4000000000000001, text=' Cześć Jane, na imię Michał.', type='transcript.text.segment')]

In [8]:
example_dir = Path("../data/example_1")

reference_asr_data = ASRData.from_txt_file(example_dir / "asr_data.txt")
print("reference:")
print(reference_asr_data)

asr_data = ASRData.from_openai_diarization(transcription_4, concat=True)
asr_data.override_speakers(["Jane", "Michał"])
print("\nModel output:")
print(asr_data)

results = calculate_csr_metrics(asr_data, reference_asr_data)
print(results)

reference:
[0.03 - 1.42] Jane: Hello I'm Jane!
[1.80 - 3.47] Michał: Cześć Jane, mam na imię Michał!

Model output:
[0.00 - 1.10] Jane:  Hello,  I'm Jane.
[1.50 - 3.40] Michał:  Cześć, Jane.  Mam na imię Michał.

         CSR results
--------------------------------
 WER (Word Error Rate):    0.00
 MER (Match Error Rate):   0.00
 WDER (Diarization Error): 0.00

